In [2]:
#domain_db.py
import sqlite3
import pandas as pd
from urllib.parse import urlsplit

class DomainDB:
    def __init__(self, db_path='domains.db'):
        self.conn = sqlite3.connect(db_path, check_same_thread=False)
    
    @classmethod
    def create_from_csv(cls, csv_path, db_path='domains.db'):
        conn = sqlite3.connect(db_path)
        df = pd.read_csv(csv_path, header=None, usecols=[1], names=['domain'])
        
        conn.execute('CREATE TABLE domains (domain TEXT PRIMARY KEY)')
        conn.executemany('INSERT INTO domains VALUES (?)', [(d,) for d in df['domain']])
        conn.commit()
        conn.close()
        return cls(db_path)
    
    def _get_host(self, url):
        try:
            u = url if '://' in url else 'http://' + url
            return urlsplit(u).hostname.lower()
        except:
            return ""
    
    def _exists(self, domain):
        return self.conn.execute('SELECT 1 FROM domains WHERE domain=?', (domain,)).fetchone() is not None
    
    def is_trusted(self, url):
        host = self._get_host(url)
        if not host:
            return False
        
        h = host[4:] if host.startswith('www.') else host
        parts = h.split('.')
        for i in range(len(parts) - 1):
            if self._exists('.'.join(parts[i:])):
                return True
        return False

# 직접 실행하면 DB 생성
if __name__ == '__main__':
    DomainDB.create_from_csv('top-1m.csv', 'domains.db')
    print("DB 생성 완료")

DB 생성 완료
